# CrewAI Mini-Capstone Project: Restaurant Social Media Marketing Pipeline

## Scenario

**Spice Fusion Bistro** is launching a new menu of **Indian/Pakistani fusion dishes** and
wants a marketing blog post plus social media content to promote it.

You will build a **5-agent sequential CrewAI pipeline** that:

1. **Culinary Trend Researcher** — discovers the new fusion dishes on the menu
2. **Nutrition Analyst** — looks up nutrition facts for each dish
3. **Local Sourcing Specialist** — identifies which ingredients are locally/seasonally sourced
4. **Blog Content Writer** — synthesizes all of the above into a publish-ready blog post
5. **Social Media Strategist** — repurposes the blog post into a Facebook post and a Twitter/X thread

Each agent (except the last two) is backed by a **tool** that queries a small mock SQLite
database (`restaurant_marketing.db`) containing:

- `dishes` — the new fusion menu items
- `nutrition` — nutrition facts per dish
- `produce` — local/regional sourcing info per ingredient

This is a **pure CrewAI** project: no LangGraph, no `main.py`, no `crewai run`. Everything
runs top-to-bottom in this notebook via plain `Crew(...).kickoff()`.

**Note on formatting:** the blog and social media tasks are written to produce three
clearly labeled sections — `## Blog Post`, `## Facebook Post`, `## Twitter Thread` — even
though this notebook doesn't parse them apart programmatically. Labeling sections
explicitly like this is good practice regardless of whether something downstream reads
them: it makes the output easy for a human to scan, and it's the same pattern you'd rely
on if you later fed this output into another system (e.g. a LangGraph node) that *does*
need to split it apart.

## Solution Notebook

## Section 1 — Environment Setup


In [ ]:
import os
import sqlite3

from crewai import Agent, Task, Crew, Process
from crewai.tools import tool

# Make sure your OPENAI_API_KEY (or other LLM provider key) is set in the environment
# before running this notebook, e.g.:
# os.environ["OPENAI_API_KEY"] = "sk-..."


## Section 2 — Mock Database Setup

We simulate the restaurant's internal systems with three SQLite tables:

- `dishes(id, name, cuisine, main_ingredients, description)`
- `nutrition(dish_id, calories, protein_g, carbs_g, fat_g, fiber_g, notes)`
- `produce(ingredient, region, season, local_farm_source)`


In [ ]:
DB_PATH = "restaurant_marketing.db"


def setup_database():
    """Create and populate the mock restaurant marketing database."""
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE dishes (
            id INTEGER PRIMARY KEY,
            name TEXT,
            cuisine TEXT,
            main_ingredients TEXT,
            description TEXT
        )
    """)

    cursor.execute("""
        CREATE TABLE nutrition (
            dish_id INTEGER,
            calories INTEGER,
            protein_g REAL,
            carbs_g REAL,
            fat_g REAL,
            fiber_g REAL,
            notes TEXT,
            FOREIGN KEY (dish_id) REFERENCES dishes (id)
        )
    """)

    cursor.execute("""
        CREATE TABLE produce (
            ingredient TEXT,
            region TEXT,
            season TEXT,
            local_farm_source TEXT
        )
    """)

    dishes = [
        (1, "Butter Chicken Tacos", "Fusion",
         "chicken, tomato, cream, corn tortilla, cilantro",
         "Classic North Indian butter chicken folded into soft corn tortillas, "
         "finished with a cilantro-mint crema."),
        (2, "Karachi Biryani Arancini", "Fusion",
         "basmati rice, lamb, saffron, breadcrumbs, mozzarella",
         "Crispy fried biryani rice balls stuffed with slow-cooked lamb and a "
         "molten mozzarella center."),
        (3, "Lahori Chapli Burger", "Fusion",
         "beef, pomegranate seeds, coriander, brioche bun",
         "A spiced Peshawari-style beef patty with pomegranate crunch, served on "
         "a toasted brioche bun."),
        (4, "Amritsari Kulcha Pizza", "Fusion",
         "flatbread dough, spiced potato, paneer, mint chutney",
         "Amritsari kulcha reimagined as a stone-baked flatbread pizza topped with "
         "spiced potato and paneer."),
        (5, "Nihari Ramen", "Fusion",
         "slow-braised beef shank, ramen noodles, chili oil, soft egg",
         "Overnight-braised Nihari broth ladled over ramen noodles, topped with "
         "chili oil and a soft-boiled egg."),
        (6, "Chana Chaat Bruschetta", "Fusion",
         "chickpeas, tamarind, toasted baguette, yogurt, sev",
         "Tangy chickpea chaat piled onto toasted baguette slices, drizzled with "
         "tamarind and yogurt."),
    ]
    cursor.executemany("INSERT INTO dishes VALUES (?, ?, ?, ?, ?)", dishes)

    nutrition = [
        (1, 420, 28.0, 32.0, 19.0, 3.0, "Good source of protein; moderate calorie dish"),
        (2, 380, 16.0, 30.0, 21.0, 1.5, "Rich and indulgent; best as a shareable appetizer"),
        (3, 540, 30.0, 38.0, 27.0, 4.0, "High protein; pomegranate adds antioxidants"),
        (4, 460, 14.0, 52.0, 20.0, 5.0, "Vegetarian; good fiber from potato and flatbread"),
        (5, 610, 34.0, 48.0, 29.0, 2.5, "Hearty and calorie-dense; great cold-weather dish"),
        (6, 240, 9.0, 34.0, 7.0, 8.0, "High fiber, lighter option, vegetarian"),
    ]
    cursor.executemany("INSERT INTO nutrition VALUES (?, ?, ?, ?, ?, ?, ?)", nutrition)

    produce = [
        ("Tomato", "Local Valley Farms", "Summer", "Green Acres Co-op"),
        ("Cilantro", "Local Valley Farms", "Year-round (greenhouse)", "Green Acres Co-op"),
        ("Basmati Rice", "Imported - Punjab region", "Year-round", "Punjab Grain Importers"),
        ("Pomegranate", "Local Valley Farms", "Fall", "Red Hill Orchards"),
        ("Potato", "Local Valley Farms", "Fall/Winter", "Green Acres Co-op"),
        ("Paneer", "Local Dairy", "Year-round", "Meadowbrook Dairy"),
        ("Chickpeas", "Regional", "Year-round (dried)", "Heartland Pulses"),
        ("Mint", "Local Valley Farms", "Spring/Summer", "Green Acres Co-op"),
    ]
    cursor.executemany("INSERT INTO produce VALUES (?, ?, ?, ?)", produce)

    conn.commit()
    conn.close()
    print("Database setup complete.")


setup_database()


## Section 3 — Tools

Tools are how agents reach into the mock database. Each tool wraps a small, focused
SQL query and returns a plain-text summary the LLM can reason over.


In [ ]:
@tool("Trending Dishes Lookup")
def get_trending_dishes() -> str:
    """Returns the restaurant's new/trending Indian and Pakistani fusion dishes,
    including cuisine influence, main ingredients, and a short description."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT name, cuisine, main_ingredients, description FROM dishes")
    rows = cursor.fetchall()
    conn.close()

    lines = []
    for name, cuisine, ingredients, desc in rows:
        lines.append(f"- {name} ({cuisine}): {desc} [Ingredients: {ingredients}]")
    return "\n".join(lines)


@tool("Nutrition Info Lookup")
def get_nutrition_info(dish_name: str) -> str:
    """Given a dish name (or partial name), returns its nutrition facts:
    calories, protein, carbs, fat, and fiber."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute(
        """
        SELECT d.name, n.calories, n.protein_g, n.carbs_g, n.fat_g, n.fiber_g, n.notes
        FROM dishes d
        JOIN nutrition n ON d.id = n.dish_id
        WHERE LOWER(d.name) LIKE LOWER(?)
        """,
        (f"%{dish_name}%",),
    )
    rows = cursor.fetchall()
    conn.close()

    if not rows:
        return f"No nutrition data found for '{dish_name}'."

    lines = []
    for name, cal, protein, carbs, fat, fiber, notes in rows:
        lines.append(
            f"{name}: {cal} kcal, {protein}g protein, {carbs}g carbs, "
            f"{fat}g fat, {fiber}g fiber. Note: {notes}"
        )
    return "\n".join(lines)


@tool("Local Produce Sourcing Lookup")
def get_local_produce_info() -> str:
    """Returns sourcing details for key ingredients: region, season, and
    local farm/supplier."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT ingredient, region, season, local_farm_source FROM produce")
    rows = cursor.fetchall()
    conn.close()

    lines = []
    for ingredient, region, season, source in rows:
        lines.append(f"- {ingredient}: sourced from {region} ({source}), in season: {season}")
    return "\n".join(lines)


## Section 4 — Agents

Five agents, each with a narrow role, a clear goal, and a backstory that shapes tone.
Only the first three need tools — the writer and strategist work purely from the
context passed to them by earlier tasks.


In [ ]:
trend_researcher = Agent(
    role="Culinary Trend Researcher",
    goal="Identify and describe the restaurant's newest Indian/Pakistani fusion "
         "dishes for a marketing blog",
    backstory=(
        "You are a food journalist specializing in South Asian cuisine trends. "
        "You have a sharp eye for what makes a dish exciting to home cooks and "
        "diners alike, and you translate menu items into engaging descriptions."
    ),
    tools=[get_trending_dishes],
    verbose=True,
)

nutrition_analyst = Agent(
    role="Nutrition Analyst",
    goal="Provide accurate, easy-to-understand nutrition information for each "
         "featured dish",
    backstory=(
        "You are a registered dietitian who consults for restaurants. You "
        "translate nutrition facts into practical, non-judgmental language that "
        "helps diners make informed choices."
    ),
    tools=[get_nutrition_info],
    verbose=True,
)

sourcing_specialist = Agent(
    role="Local Sourcing Specialist",
    goal="Highlight where key ingredients come from, emphasizing local and "
         "seasonal sourcing",
    backstory=(
        "You are the restaurant's supply chain coordinator with deep "
        "relationships with regional farms and importers. You care about "
        "telling the story of where the food comes from."
    ),
    tools=[get_local_produce_info],
    verbose=True,
)

blog_writer = Agent(
    role="Blog Content Writer",
    goal="Write an engaging, well-structured restaurant blog post combining "
         "dish trends, nutrition, and sourcing",
    backstory=(
        "You are a restaurant marketing copywriter who has written dozens of "
        "high-performing blog posts. You know how to weave factual detail into "
        "a warm, appetizing narrative."
    ),
    verbose=True,
)

social_media_strategist = Agent(
    role="Social Media Strategist",
    goal="Repurpose the blog post into platform-native Facebook and Twitter/X "
         "content that drives restaurant visits",
    backstory=(
        "You are a social media manager for restaurant brands. You know "
        "Facebook rewards warm, story-driven posts with photos, while "
        "Twitter/X rewards short, punchy hooks and threads."
    ),
    verbose=True,
)


## Section 5 — Tasks

Tasks are chained with `context=[...]` so each downstream task receives the outputs
of the tasks it depends on. This is the same context-passing pattern used in the
bond trading pipeline (market data -> position -> risk -> threshold -> decision).


In [ ]:
research_task = Task(
    description=(
        "Research the restaurant's new Indian/Pakistani fusion dishes. For each "
        "dish, capture the name, cuisine influence, main ingredients, and what "
        "makes it noteworthy. Use the trending dishes tool."
    ),
    expected_output="A list of the new dishes with a 1-2 sentence hook for each.",
    agent=trend_researcher,
)

nutrition_task = Task(
    description=(
        "For each dish identified by the Culinary Trend Researcher, look up its "
        "nutrition facts (calories, protein, carbs, fat, fiber) and write a "
        "short, friendly nutrition summary."
    ),
    expected_output="A nutrition summary for each dish written in approachable language.",
    agent=nutrition_analyst,
    context=[research_task],
)

sourcing_task = Task(
    description=(
        "Using the dishes and their main ingredients, identify which "
        "ingredients are locally/seasonally sourced and describe the local "
        "farms or suppliers behind them."
    ),
    expected_output="A short write-up per dish (or per key ingredient) on local/seasonal sourcing.",
    agent=sourcing_specialist,
    context=[research_task],
)

blog_task = Task(
    description=(
        "Write a complete restaurant blog post (600-900 words) titled something "
        "like 'New on the Menu: Indian & Pakistani Fusion Dishes You Have to "
        "Try'. Combine the dish descriptions, nutrition info, and local "
        "sourcing story into one cohesive, appetizing narrative with an intro, "
        "a section per dish, and a closing call-to-action to visit the "
        "restaurant. Put the entire blog post under a Markdown heading exactly "
        "'## Blog Post'."
    ),
    expected_output=(
        "A polished, publish-ready blog post starting with the exact heading "
        "'## Blog Post', followed by Markdown content with headers per dish."
    ),
    agent=blog_writer,
    context=[research_task, nutrition_task, sourcing_task],
)

social_media_task = Task(
    description=(
        "Based on the finished blog post, create two clearly separated "
        "sections: "
        "(1) a Facebook post (150-200 words, warm tone, includes a call to "
        "action and 3-5 hashtags) under the exact heading '## Facebook Post', "
        "and "
        "(2) a Twitter/X thread of 4-5 tweets (each under 280 characters) "
        "under the exact heading '## Twitter Thread', teasing the new dishes, "
        "with a final tweet linking back to the blog and a call to action. "
        "Use these exact headings so the two sections are easy to tell apart."
    ),
    expected_output=(
        "Two sections, each starting with its exact Markdown heading: "
        "'## Facebook Post' and '## Twitter Thread'."
    ),
    agent=social_media_strategist,
    context=[blog_task],
)


## Section 6 — Assemble and Run the Crew


In [ ]:
crew = Crew(
    agents=[
        trend_researcher,
        nutrition_analyst,
        sourcing_specialist,
        blog_writer,
        social_media_strategist,
    ],
    tasks=[
        research_task,
        nutrition_task,
        sourcing_task,
        blog_task,
        social_media_task,
    ],
    process=Process.sequential,
    verbose=True,
)

result = crew.kickoff()


## Section 7 — Review the Output

Run the cell below after `kickoff()` completes to inspect the final deliverable:
the blog post and social media content produced by the last task in the pipeline.


In [ ]:
print(result)
